# Hone SN5 ARC Miner Lab

This notebook is for cheap research only. Do not paste wallet keys, seed phrases, Bittensor private keys, or paid API keys into Colab.

In [ ]:
#@title Configuration
REPO_URL = "https://github.com/YOUR_USER/hone-arc-miner"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
N_PROBLEMS = 10  #@param {type:"integer"}
SEED = 7  #@param {type:"integer"}
CHAIN_MIN = 3  #@param {type:"integer"}
CHAIN_MAX = 7  #@param {type:"integer"}

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORKDIR = Path("/content")
for name in ("hone-arc-miner", "research-hone"):
    path = WORKDIR / name
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "hone-arc-miner"], cwd=WORKDIR, check=True)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/manifold-inc/hone", "research-hone"], cwd=WORKDIR, check=True)
os.chdir(WORKDIR / "hone-arc-miner")
print(Path.cwd())

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "solver/requirements.txt"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "miner-server/requirements.txt"], check=True)
subprocess.run(["python", "-m", "py_compile", "solver/arc_main.py", "solver/arc_inference_phase.py", "solver/arc_prep_phase.py", "solver/arc_solver.py", "solver/arc_utils.py", "tools/evaluate_local.py", "miner-server/server.py"], check=True)
print("compile ok")

In [ ]:
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
json_out = f"/content/hone_eval_n{N_PROBLEMS}_seed{SEED}.json"
cmd = [
    "python", "tools/evaluate_local.py",
    "--n", str(N_PROBLEMS),
    "--seed", str(SEED),
    "--chain-min", str(CHAIN_MIN),
    "--chain-max", str(CHAIN_MAX),
    "--json-out", json_out,
]
subprocess.run(cmd, check=True)
print("report:", json_out)

In [ ]:
import json
from pathlib import Path

report = json.loads(Path(json_out).read_text())
summary = {k: report[k] for k in [
    "n", "seed", "chain_min", "chain_max", "exact_matches", "exact_match_rate",
    "shape_match_rate", "avg_partial_correctness", "avg_grid_similarity", "elapsed_s",
]}
summary